# H-Node Hallucination Detection — Colab Pipeline

End-to-end run of the H-Node hallucination detector built on vLLM-Hook. Three stages:

1. **extract** — capture last-token hidden states at every transformer layer for 600 labeled TruthfulQA prompts (~5–10 min on T4)
2. **train** — fit per-layer logistic-regression probes, pick the best layer, identify top-50 H-Nodes (~30 sec, CPU)
3. **detect** — load only the best layer and score new prompts (~1 min on T4)

**Important:** we pin `vllm==0.7.3` because Colab's default vLLM (v0.21+) enables async scheduling, which the hook framework's `execute_model` override doesn't yet support.

After installs, **Runtime → Restart Runtime** (mandatory) before running the rest.

## 1. GPU sanity check

In [2]:
!nvidia-smi

Sat May 16 12:12:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the fork

In [3]:
%cd /content
!rm -rf vLLM-Hook
!git clone https://github.com/Samarpit-bhatia/vLLM-Hook.git
%cd /content/vLLM-Hook
!git log --oneline -5

/content
Cloning into 'vLLM-Hook'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 297 (delta 85), reused 90 (delta 57), pack-reused 132 (from 2)
Receiving objects: 100% (297/297), 2.15 MiB | 23.97 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/vLLM-Hook
5ccb320 (HEAD -> main, origin/main, origin/HEAD) fix(halludetect): disable async scheduling for v0.21+ compatibility
e8d55f9 fix(halludetect): add project root to sys.path
be2c062 add H-Node hallucination detection pipeline
631b0f9 Merge pull request #13 from tburleyinfo/colab-notebooks-upstream
7f58b15 Address Colab notebook review feedback


## 3. Install dependencies (with vLLM pin)

After this cell finishes, **restart the runtime** (Runtime → Restart runtime) before running anything else. Python has cached the wrong vllm and won't pick up the new one without a restart.

In [ ]:
# Pin to a vLLM the hook framework was tested against (pre-async-scheduling)
# AND pin transformers to a version compatible with that vLLM. vllm 0.7.3
# uses tokenizer.all_special_tokens_extended which was removed in
# transformers >= 4.50, so we pin to 4.49.0.
!pip uninstall -y vllm transformers 2>/dev/null || true
!pip install "vllm==0.7.3" "transformers==4.49.0" --quiet
!pip install -e /content/vLLM-Hook/vllm_hook_plugins --quiet
!pip install datasets scikit-learn --quiet
print("\n>>> NOW RESTART THE RUNTIME (Runtime → Restart runtime), then continue from section 4.")

## 4. Verify environment (after restart)

In [ ]:
import torch, vllm, transformers
print("torch:       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("vllm:        ", vllm.__version__)
print("transformers:", transformers.__version__)
print("gpu:         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(none)")
assert vllm.__version__.startswith("0.7"), f"Expected vllm 0.7.x, got {vllm.__version__} — restart runtime?"
assert transformers.__version__.startswith("4.49"), f"Expected transformers 4.49.x, got {transformers.__version__} — restart runtime?"

## 5. Stage 1 — Extract (forward pass with hooks)

Loads Qwen2.5-3B, registers forward hooks on all 36 transformer layers via `ProbeHiddenStatesWorker`, runs ~600 TruthfulQA prompts, dumps last-token activations per layer to `hallucination_detection/artifacts/activations.pt`.

Total time on T4: ~10 min (mostly model load + warmup; the actual prompt loop is fast).

In [3]:
%cd /content/vLLM-Hook
!git pull
!rm -rf /dev/shm/vllm_hook
!python examples/demo_halludetect.py extract

/content/vLLM-Hook
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 5.44 KiB | 2.72 MiB/s, done.
From https://github.com/Samarpit-bhatia/vLLM-Hook
   5ccb320..ac29f8c  main       -> origin/main
Updating 5ccb320..ac29f8c
Fast-forward
 examples/demo_halludetect.py |  10 +-
 implement.ipynb              | 362 +++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 370 insertions(+), 2 deletions(-)
 create mode 100644 implement.ipynb
Building TruthfulQA labeled pairs...
  600 prompts (300 grounded / 300 hallucinated)
Loading model under ProbeHiddenStatesWorker (training config)...
INFO 05-16 12:29:44 __init__.py:207] Automatically detected platform cuda.
INFO 05-16 12:29:46 __init__.py:30] Available plugins for group vllm.general_plugins:
INFO 05-16 12:29:46 __init__.py:32] name=hook_registry, value=vllm_h

In [ ]:
# Confirm the extracted bundle exists and has the expected shape
import torch, os
p = "hallucination_detection/artifacts/activations.pt"
assert os.path.exists(p), f"Missing {p} — extract stage failed"
bundle = torch.load(p, map_location="cpu")
print("layers captured:", len(bundle["activations"]))
print("hidden size:    ", next(iter(bundle["activations"].values())).shape[1])
print("prompts:        ", len(bundle["labels"]))
print("grounded/hall:  ", int((bundle["labels"] == 0).sum()), "/", int((bundle["labels"] == 1).sum()))

## 6. Stage 2 — Train (probe fit, best-layer pick, H-Node ID)

Pure CPU (sklearn). Rewrites the inference config to capture only the chosen best layer.

In [ ]:
!python examples/demo_halludetect.py train

In [ ]:
# Inspect the trained probe
import json, numpy as np
with open("hallucination_detection/artifacts/probe.json") as f:
    meta = json.load(f)
print("best_layer:", meta["best_layer"])
print("AUC@best:  ", meta["auc_per_layer"][str(meta["best_layer"])])
print("n_h_nodes: ", meta["n_h_nodes"])

data = np.load("hallucination_detection/artifacts/probe.npz")
print("H-Nodes (first 10):", data["h_node_indices"][:10].tolist())

## 7. Stage 3 — Detect (live scoring)

Reloads the model with the *inference* config (only the best layer is hooked), then scores 6 example prompts: grounded answers should print P(hallucinated) near 0, wrong answers near 1.

In [ ]:
!rm -rf /dev/shm/vllm_hook   # fresh state
!python examples/demo_halludetect.py detect

## 8. Save artifacts back to your machine

In [ ]:
from google.colab import files
files.download("hallucination_detection/artifacts/probe.npz")
files.download("hallucination_detection/artifacts/probe.json")